# 01 - AOI & Data Acquisition

Defines the 8 wheat-growing state AOIs (FAO GAUL), Rabi season windows, and acquires **Sentinel-1 SAR**, **Sentinel-2 optical** (cloud-masked), and **MODIS NDVI/LST**. Also demonstrates local ingestion of **Resourcesat-2 AWiFS** GeoTIFFs and **IMD** gridded NetCDF.

> Prerequisite: `pip install -r ../requirements.txt` and a Google Earth Engine account (`ee.Authenticate()`).

In [ ]:
import sys, yaml
sys.path.append('..')
from src import gee_utils, io_utils

with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)
STATES = cfg['states']
STATES

In [ ]:
import ee
gee_utils.init_ee()  # first run triggers ee.Authenticate()
states_fc = gee_utils.get_states_fc(STATES)
aoi = states_fc.geometry()
print('States loaded:', states_fc.size().getInfo())

## Rabi season window

Sowing starts ~1 November; harvest ends ~30 April. Early-season optical data is cloud-limited, hence the **SAR-first** strategy.

In [ ]:
SEASON_YEAR = 2023  # Rabi 2023-24
start = f'{SEASON_YEAR}-11-01'
end = f'{SEASON_YEAR + 1}-04-30'

s1 = gee_utils.get_s1_collection(aoi, start, end)
s2 = gee_utils.get_s2_collection(aoi, start, end, cfg['sensors']['sentinel2']['max_cloud_prob'])
print('Sentinel-1 scenes:', s1.size().getInfo())
print('Sentinel-2 scenes (cloud-masked):', s2.size().getInfo())

In [ ]:
modis_ndvi = gee_utils.get_modis_ndvi(aoi, start, end)
modis_lst = gee_utils.get_modis_lst(aoi, start, end)
print('MODIS NDVI composites:', modis_ndvi.size().getInfo())
print('MODIS LST composites:', modis_lst.size().getInfo())

In [ ]:
import geemap
m = geemap.Map(center=[26.5, 78.0], zoom=5)
m.addLayer(states_fc.style(color='black', fillColor='00000000'), {}, 'States')
m.addLayer(s2.median().select(['B4', 'B3', 'B2']), {'min': 0, 'max': 0.3}, 'S2 median RGB')
m.addLayer(s1.median().select('VH'), {'min': -25, 'max': -5}, 'S1 VH median', False)
m

## Local data: AWiFS / LISS-III and IMD

**Caution:** AWiFS 56 m resolution causes mixed-pixel anomalies on smallholder plots - use it for wide-swath state coverage, Sentinel-2 for plot-level work.

In [ ]:
# AWiFS GeoTIFF ingestion (uncomment with real paths)
# arr, profile = io_utils.load_awifs_geotiff('path/to/awifs_scene.tif')
# ndvi = io_utils.awifs_ndvi(arr[1], arr[2])  # band2=red, band3=NIR

# IMD gridded NetCDF (0.25 deg rainfall / 1 deg temperature)
# rain = io_utils.load_imd_netcdf('path/to/imd_rainfall.nc', var='RAINFALL')
print('Replace paths above with real AWiFS / IMD files when available.')